In [1]:
vendor_paths = {
    'cisco': 'database/cisco_oferta.csv',
    'nokia': 'database/nokia_oferta.csv',
    'arista': 'database/arista_oferta.csv',
}

import pandas as pd
import sys
from pathlib import Path
from model.modular import Modular
from model.solution import Solution
from model.switch import Switch
sys.path.insert(0, str(Path('.').resolve()))

lookup_path = Path('database/price_lookup.xlsx')
lookup_df = pd.read_excel(lookup_path, sheet_name=0)
lookup_df = lookup_df[['code', 'cost']].drop_duplicates(subset=['code'], keep='last')

for vendor_path in vendor_paths.values():
    vendor_df = pd.read_csv(vendor_path)
    if 'cost' in vendor_df.columns:
        vendor_df = vendor_df.drop(columns=['cost'])
    vendor_df = vendor_df.merge(lookup_df, on='code', how='left')
    vendor_df.to_csv(vendor_path, index=False)

generated_ports_path = 'generated_ports_custom.xlsx'

ports_df = pd.read_excel(generated_ports_path)
speed_columns = [col for col in ports_df.columns if col not in ['profile', 'role']]



In [2]:
def load_vendor_data(csv_path):
    df = pd.read_csv(csv_path)
    modules_df = df[df['type'] == 'modular']
    linecards_df = df[df['type'] == 'linecard']
    fixed_df = df[df['type'] == 'fixed']
    cost_by_code = (
        df.dropna(subset=['cost'])
        .drop_duplicates(subset=['code'])
        .set_index('code')['cost']
        .to_dict()
    )
    module_codes = modules_df['code'].unique()
    return df, modules_df, linecards_df, cost_by_code, module_codes, fixed_df


def build_requirement_from_row(row, speed_columns):
    requirement = pd.DataFrame([{
        'code': 'requirement',
        **{col: int(row[col]) for col in speed_columns}
    }])
    zero_cols = [col for col in speed_columns if requirement.at[0, col] == 0]
    if zero_cols:
        requirement = requirement.drop(columns=zero_cols)
    return requirement


def solve_lowest_cost(requirement, df, module_codes, cost_by_code, fixed_codes):
    best_result = None
    best_cost = None
    best_module = None
    for module_code in module_codes:
        try:
            module_data = df[df['code'] == module_code]
            module_family = module_data['family'].iloc[0]
            linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
            combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

            modular = Modular(combined_data)
            solution = Solution(modular, requirement)
            result = solution.solve(heuristic="H2")

            if result is None or result.empty:
                continue

            module_cost = cost_by_code.get(module_code, 0)
            total_cost = result['code'].map(cost_by_code).fillna(0).sum() + module_cost
            if best_cost is None or total_cost < best_cost:
                best_cost = total_cost
                best_result = result
                best_module = module_code
        except Exception:
            continue
    for fixed_code in fixed_codes:
        try:
            fixed_data = df[df['code'] == fixed_code]
            if fixed_data.empty:
                continue
            fixed_switch = Switch(fixed_data)
            if not fixed_switch.check_if_satisfies(requirement):
                continue
            total_cost = fixed_switch.cost
            fixed_result = fixed_switch.obtain_max_value_configuration(requirement)
            if best_cost is None or total_cost < best_cost:
                best_cost = total_cost
                best_result = fixed_result
                best_module = fixed_code
        except Exception:
            continue
    return best_result, best_cost, best_module


summary_rows = []
for vendor, csv_path in vendor_paths.items():
    df, modules_df, linecards_df, cost_by_code, module_codes, fixed_df = load_vendor_data(csv_path)

    print(f"{vendor}: {len(modules_df)} module rows")
    print(f"{vendor}: {linecards_df['code'].nunique()} unique linecards")
    print(f"{vendor}: available modules: {modules_df['code'].unique().tolist()}")
    print(f"{vendor}: available fixed: {fixed_df['code'].unique().tolist()}")

    vendor_rows = []
    for _, row in ports_df.iterrows():
        requirement = build_requirement_from_row(row, speed_columns)
        best_result, best_cost, best_module = solve_lowest_cost(
            requirement, df, module_codes, cost_by_code, fixed_df['code'].unique()
        )

        summary = {col: int(row[col]) for col in speed_columns}
        if 'profile' in ports_df.columns:
            summary['profile'] = row['profile']
        if 'role' in ports_df.columns:
            summary['role'] = row['role']

        summary['vendor'] = vendor
        summary['best_module'] = best_module
        summary['solution_linecards'] = [] if best_result is None else best_result['code'].tolist()
        summary['solution_cost'] = None if best_cost is None else float(best_cost)

        vendor_rows.append(summary)
        summary_rows.append(summary)

    vendor_df = pd.DataFrame(vendor_rows)
    vendor_output_path = f"{vendor}_results.csv"
    vendor_df.to_csv(vendor_output_path, index=False)
    print(f"Saved {len(vendor_df)} rows to {vendor_output_path}")

summary_df = pd.DataFrame(summary_rows)

cisco: 6 module rows
cisco: 27 unique linecards
cisco: available modules: ['9808', '9804', '9516', '9508', '9504', '9400']
cisco: available fixed: ['HF6100-60L4D', 'HF6100-32D', 'HF6100-64ED', '93180YC-EX', '93108TC-EX', '93180LC-EX', '93400LD-H1', '9332D-H2R', '9364D-GX2A', '9348D-GX2A', '9332D-GX2B', 'N9K-C9316D-GX', 'N9K-C93600CD-GX', 'N9K-C9364C-GX', '93180YC-FX3', '93108TC-FX3', '93108TC-FX3P', '9348GC-FX3', '9348GC-FX3PH', '9336C-FX2', '9336C-FX2-E', '93240YC-FX2', '93360YC-FX2', '93216TC-FX2', '93180YC-FX', '93108TC-FX', '9348GC-FXP', '92348GC-X', '92348GC-FX3', '92160YC-X', '9272Q', '92304QC', '9236C', '92300YC', 'N9364E-SP2R']
Saved 1 rows to cisco_results.csv
nokia: 3 module rows
nokia: 3 unique linecards
nokia: available modules: ['7250 IXR-6e', '7250 IXR-10e', '7250  IXR-18e']
nokia: available fixed: ['7220 IXR-H2', '7220 IXR-H4-32D', '7220 IXR-H4', '7220 IXR-H5-32D', '7220 IXR-H5-64D', '7250 IXR-X1b', '7250 IXR-X3b', '7220 IXR-D5', '7220 IXR-D4', '7220 IXR-D3L', '7220 IXR-

In [3]:
import pandas as pd
import plotly.graph_objects as go
import ast
import json
from pathlib import Path

vendor1 = "nokia"
vendor2 = "arista"
vendor3 = "cisco"

# Assuming vendor_paths is defined earlier
vendor_paths = {
    'cisco': 'database/cisco_oferta.csv',
    'nokia': 'database/nokia_oferta.csv',
    'arista': 'database/arista_oferta.csv',
}

result_paths_by_vendor = {vendor: f"{vendor}_results.csv" for vendor in vendor_paths}


def _get_speed_columns(df):
    speed_cols = []
    for col in df.columns:
        if col == "solution_cost":
            continue
        try:
            float(col)
        except (TypeError, ValueError):
            continue
        speed_cols.append(col)
    return speed_cols


def _wrap_linecards(value, items_per_line=6, max_items=24, max_chars=300):
    if pd.isna(value):
        return ""
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            value = [value]
    if isinstance(value, list):
        truncated = value[:max_items]
        if len(value) > max_items:
            truncated = truncated + ["..."]
        chunks = [
            ", ".join(str(item) for item in truncated[i:i + items_per_line])
            for i in range(0, len(truncated), items_per_line)
        ]
        text = "<br>".join(chunks)
    else:
        text = str(value)
    if len(text) > max_chars:
        text = text[:max_chars] + "..."
    return text


def _load_results(path):
    return pd.read_csv(path)


def _build_compare_df(df1, df2, df3, speed_columns):
    union_index = df1.index.union(df2.index).union(df3.index)
    base = pd.DataFrame(index=union_index)

    for col in speed_columns:
        base[col] = df1[col].reindex(union_index)
        if col in df2.columns:
            base[col] = base[col].fillna(df2[col].reindex(union_index))
        if col in df3.columns:
            base[col] = base[col].fillna(df3[col].reindex(union_index))

    info_cols = ["profile", "role", "best_module", "solution_linecards"]
    for col in info_cols:
        if col in df1.columns:
            base[f"v1_{col}"] = df1[col].reindex(union_index)
        if col in df2.columns:
            base[f"v2_{col}"] = df2[col].reindex(union_index)
        if col in df3.columns:
            base[f"v3_{col}"] = df3[col].reindex(union_index)

    base["vendor1_cost"] = df1["solution_cost"].reindex(union_index)
    base["vendor2_cost"] = df2["solution_cost"].reindex(union_index)
    base["vendor3_cost"] = df3["solution_cost"].reindex(union_index)

    base["total_bandwidth"] = sum(
        base[col].fillna(0) * float(col) for col in speed_columns
    )
    return base


def _make_hover_text(row, vendor1, vendor2, vendor3, speed_columns):
    def _vendor_block(name, cost_col, prefix):
        lines = []
        cost_val = row.get(cost_col)
        if pd.notna(cost_val):
            lines.append(f"{name} cost: {cost_val}")
        for key, label in [
            ("profile", "profile"),
            ("role", "role"),
            ("best_module", "best_module"),
        ]:
            col = f"{prefix}{key}"
            val = row.get(col)
            if pd.notna(val):
                lines.append(f"{name} {label}: {val}")
        linecards_col = f"{prefix}solution_linecards"
        linecards_val = row.get(linecards_col)
        if pd.notna(linecards_val):
            lines.append(f"{name} linecards: {_wrap_linecards(linecards_val)}")
        return lines

    speed_lines = []
    for col in speed_columns:
        val = row.get(col)
        if pd.notna(val) and val != 0:
            speed_lines.append(f"{col}: {int(val)}")

    if speed_lines:
        speed_block = "ports by speed: " + ", ".join(speed_lines)
    else:
        speed_block = "ports by speed: 0"

    return "<br>".join(
        _vendor_block(vendor1, "vendor1_cost", "v1_")
        + _vendor_block(vendor2, "vendor2_cost", "v2_")
        + _vendor_block(vendor3, "vendor3_cost", "v3_")
        + [speed_block]
    )


def _generate_html_with_sliders(vendor1, vendor2, vendor3):
    df1 = _load_results(result_paths_by_vendor[vendor1])
    df2 = _load_results(result_paths_by_vendor[vendor2])
    df3 = _load_results(result_paths_by_vendor[vendor3])
    speed_columns = _get_speed_columns(df1)
    compare_df = _build_compare_df(df1, df2, df3, speed_columns)

    # Prepare all data points
    all_data = []
    for _, row in compare_df.iterrows():
        bw = float(row["total_bandwidth"]) if pd.notna(row["total_bandwidth"]) else 0
        cost1 = float(row["vendor1_cost"]) if pd.notna(row["vendor1_cost"]) else None
        cost2 = float(row["vendor2_cost"]) if pd.notna(row["vendor2_cost"]) else None
        cost3 = float(row["vendor3_cost"]) if pd.notna(row["vendor3_cost"]) else None
        hover_text = _make_hover_text(row, vendor1, vendor2, vendor3, speed_columns)

        point_data = {
            'bw': bw,
            'cost1': cost1,
            'cost2': cost2,
            'cost3': cost3,
            'hover': hover_text
        }

        # Add speed column values
        for col in speed_columns:
            if str(col) != "1600":
                point_data[f'speed_{col}'] = float(row[col]) if pd.notna(row[col]) else 0

        all_data.append(point_data)

    # Create initial figure (will be updated by JavaScript)
    fig = go.Figure()

    # Add placeholder traces
    fig.add_trace(go.Scatter(
        x=[], y=[], mode="markers",
        marker=dict(color="green", symbol="circle", size=4),
        name=f"{vendor1}",
        hovertext=[],
        hoverinfo="text"
    ))
    fig.add_trace(go.Scatter(
        x=[], y=[], mode="markers",
        marker=dict(color="red", symbol="circle", size=4),
        name=f"{vendor2}",
        hovertext=[],
        hoverinfo="text"
    ))
    fig.add_trace(go.Scatter(
        x=[], y=[], mode="markers",
        marker=dict(color="blue", symbol="circle", size=4),
        name=f"{vendor3}",
        hovertext=[],
        hoverinfo="text"
    ))

    fig.update_layout(
        title=f"{vendor1} vs {vendor2} vs {vendor3} (row-by-row costs)",
        template="plotly_white",
        paper_bgcolor="#E5ECF6",
        plot_bgcolor="#E5ECF6",
        hoverlabel=dict(font_size=10),
        height=600,
        margin=dict(l=50, r=50, t=80, b=50)
    )
    fig.update_xaxes(title="total_bandwidth")
    fig.update_yaxes( title="solution_cost")

    # Get slider configurations
    slider_configs = {}
    for col in speed_columns:
        if str(col) != "1600" and col in compare_df.columns:
            max_val = int(compare_df[col].max()) if compare_df[col].notna().any() else 0
            slider_configs[col] = {'min': 0, 'max': max_val}

    # Generate HTML with embedded JavaScript
    html_content = fig.to_html(include_plotlyjs='cdn', div_id='plotly-div')

    # Build slider HTML
    sliders_html = '<div style="display: flex; flex-wrap: wrap; gap: 20px; padding: 20px; background-color: #ffffff; border-radius: 8px; margin-bottom: 20px;">'

    for col, config in slider_configs.items():
        sliders_html += f'''
        <div style="display: flex; flex-direction: column; align-items: center; min-width: 120px;">
            <label style="font-weight: bold; margin-bottom: 8px; font-size: 14px;">{col}</label>
            <div style="display: flex; flex-direction: column; gap: 8px; width: 100%;">
                <div style="display: flex; align-items: center; gap: 8px;">
                    <label style="font-size: 12px; min-width: 35px;">Min:</label>
                    <input type="range" id="slider-{col}-min" min="{config['min']}" max="{config['max']}" value="{config['min']}"
                           style="flex: 1;" oninput="updateSlider('{col}', 'min', this.value)">
                    <span id="value-{col}-min" style="min-width: 40px; font-size: 12px;">{config['min']}</span>
                </div>
                <div style="display: flex; align-items: center; gap: 8px;">
                    <label style="font-size: 12px; min-width: 35px;">Max:</label>
                    <input type="range" id="slider-{col}-max" min="{config['min']}" max="{config['max']}" value="{config['max']}"
                           style="flex: 1;" oninput="updateSlider('{col}', 'max', this.value)">
                    <span id="value-{col}-max" style="min-width: 40px; font-size: 12px;">{config['max']}</span>
                </div>
            </div>
        </div>
        '''

    sliders_html += '</div>'

    # JavaScript for filtering
    js_code = f'''
    <script>
        // Store all data points
        const allData = {json.dumps(all_data)};
        const sliderRanges = {json.dumps(slider_configs)};

        function updateSlider(col, minOrMax, value) {{
            // Update display value
            document.getElementById('value-' + col + '-' + minOrMax).textContent = value;

            // Update the plot
            filterAndUpdatePlot();
        }}

        function filterAndUpdatePlot() {{
            // Get current slider values
            const filters = {{}};
            for (const col in sliderRanges) {{
                const minVal = parseFloat(document.getElementById('slider-' + col + '-min').value);
                const maxVal = parseFloat(document.getElementById('slider-' + col + '-max').value);
                filters[col] = {{min: minVal, max: maxVal}};
            }}

            // Filter data
            const filteredData = allData.filter(point => {{
                for (const col in filters) {{
                    const speedKey = 'speed_' + col;
                    const value = point[speedKey] || 0;
                    if (value < filters[col].min || value > filters[col].max) {{
                        return false;
                    }}
                }}
                return true;
            }});

            // Prepare data for each trace
            const greenX = [], greenY = [], greenText = [];
            const redX = [], redY = [], redText = [];
            const blueX = [], blueY = [], blueText = [];

            filteredData.forEach(point => {{
                const bw = point.bw;
                const cost1 = point.cost1;
                const cost2 = point.cost2;
                const cost3 = point.cost3;
                const hover = point.hover;

                if (cost1 !== null) {{
                    greenX.push(bw);
                    greenY.push(cost1);
                    greenText.push(hover);
                }}
                if (cost2 !== null) {{
                    redX.push(bw);
                    redY.push(cost2);
                    redText.push(hover);
                }}
                if (cost3 !== null) {{
                    blueX.push(bw);
                    blueY.push(cost3);
                    blueText.push(hover);
                }}
            }});

            // Update Plotly traces
            Plotly.restyle('plotly-div', {{
                x: [greenX, redX, blueX],
                y: [greenY, redY, blueY],
                hovertext: [greenText, redText, blueText]
            }}, [0, 1, 2]);
        }}

        // Initial plot render
        window.addEventListener('load', function() {{
            filterAndUpdatePlot();
        }});
    </script>
    '''

    # Combine everything
    full_html = f'''
    <!DOCTYPE html>
    <html>
    <head>
        <title>{vendor1} vs {vendor2} vs {vendor3} Comparison</title>
        <meta charset="utf-8">
        <style>
            body {{
                font-family: Arial, sans-serif;
                margin: 20px;
            }}
            h1 {{
                text-align: center;
                color: #333;
            }}
        </style>
    </head>
    <body>
        <h1>{vendor1} vs {vendor2} vs {vendor3}</h1>
        {sliders_html}
        {html_content}
        {js_code}
    </body>
    </html>
    '''

    output_path = f"{vendor1}_vs_{vendor2}_vs_{vendor3}_comparison_custom.html"
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(full_html)

    print(f"Generated: {output_path}")
    return output_path


# Generate the standalone HTML
_generate_html_with_sliders(vendor1, vendor2, vendor3)


Generated: nokia_vs_arista_vs_cisco_comparison_custom.html


'nokia_vs_arista_vs_cisco_comparison_custom.html'